# ARC Prize 2026: ARC-AGI-3 LCLD Agent Version 10.0

**Architecture**: Neuro-Symbolic Tri-Agent (Explorer, DSL Coder, Solver) with Brusentsov Ternary Logic,
Isolated Memory Contours (ISO-1..ISO-5), Deterministic ARGALite Perception, and Tufa Single-RESET Protection.
**Model**: Qwen3.8 27B FP8 (`foysalemonshanto/qwen3-8-27b-fp8-repacked-v1`) via vLLM with FlashAttention.
**Configuration**: Concurrency=5, Thinking=32K tokens, Context=64K..128K, HeavySmoke=True.


In [ ]:
# =============================================================================
# CELL 1: OFFLINE COMPETITION RUNTIME INSTALLATION
# =============================================================================
import subprocess, sys, os, pathlib

print('=== Installing ARC-AGI competition wheels ===', flush=True)
# Install competition runtime wheels (arc-agi, arcengine)
# Must be installed strictly from the competition directory with --no-deps
# to prevent overriding pre-installed Kaggle packages (such as Pillow).
candidate_comp_dirs = [
    '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels',
    '/kaggle/input/arc-prize-2026-arc-agi-3/arc_agi_3_wheels',
]
comp_dir = next((p for p in candidate_comp_dirs if os.path.isdir(p)), None)
if comp_dir:
    print(f'Found competition wheels directory: {comp_dir}', flush=True)
    for pkg in ['arcengine', 'arc-agi']:
        cmd = [sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', f'--find-links={comp_dir}', pkg]
        res = subprocess.run(cmd, capture_output=True, text=True)
        if res.returncode == 0:
            print(f'[OK] Installed {pkg}', flush=True)
        else:
            print(f'Notice during {pkg} install: {res.stderr[-500:] if res.stderr else res.stdout[-500:]}', flush=True)
else:
    print('Notice: Competition wheels directory not found, assuming pre-installed.', flush=True)

print('Environment initialization complete.', flush=True)


In [ ]:
# =============================================================================
# CELL 2: UNPACK LCLD V10 AGENT PAYLOAD
# =============================================================================
import base64, io, zipfile, pathlib, sys

PAYLOAD_B64 = '...'

DEPLOY_DIR = pathlib.Path('/tmp/arc_lcld_agent/Code')
DEPLOY_DIR.mkdir(parents=True, exist_ok=True)

zip_data = base64.b64decode(PAYLOAD_B64)
with zipfile.ZipFile(io.BytesIO(zip_data)) as zf:
    zf.extractall(DEPLOY_DIR)

if str(DEPLOY_DIR) not in sys.path:
    sys.path.insert(0, str(DEPLOY_DIR))

print(f'Successfully deployed {len(zip_data)} bytes to {DEPLOY_DIR}', flush=True)


In [ ]:
# =============================================================================
# CELL 3: PHASE-A STRUCTURAL PREFLIGHT & SUBMISSION ARTIFACT ASSURANCE
# =============================================================================
import os, pathlib, json
import pandas as pd
import lcld_preflight

# Run structural preflight test (deterministic offline verification)
lcld_preflight.run_preflight()

# Ensure /kaggle/working/submission.parquet exists for Kaggle evaluator
working_root = pathlib.Path('/kaggle/working')
working_root.mkdir(parents=True, exist_ok=True)
submission_path = working_root / 'submission.parquet'
if not submission_path.exists():
    dummy_submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'],
    )
    dummy_submission.to_parquet(submission_path, index=False)
    print(f'Created required competition submission artifact at {submission_path}', flush=True)

is_rerun = os.getenv('KAGGLE_IS_COMPETITION_RERUN', '').strip().lower() in ('1', 'true')
print(f'KAGGLE_IS_COMPETITION_RERUN = {is_rerun}', flush=True)


In [ ]:
# =============================================================================
# CELL 4: PHASE-A HEAVY COMBAT SMOKE TEST (vLLM & QWEN3.8 DIAGNOSTICS)
# =============================================================================
ENABLE_PHASE_A_HEAVY_SMOKE = True
import os, pathlib, json
import pandas as pd

is_rerun = os.getenv('KAGGLE_IS_COMPETITION_RERUN', '').strip().lower() in ('1', 'true')
working_root = pathlib.Path('/kaggle/working')
submission_path = working_root / 'submission.parquet'

if not is_rerun:
    if ENABLE_PHASE_A_HEAVY_SMOKE:
        print('=== Launching Phase-A Heavy Combat Smoke Diagnostics ===', flush=True)
        try:
            import phase_a_heavy_smoke
            smoke_summary = phase_a_heavy_smoke.run_phase_a_smoke_pipeline()
            print(f'Heavy smoke execution status: {smoke_summary.get("status")}', flush=True)
        except Exception as exc:
            print(f'[HEAVY-SMOKE WARNING] Caught smoke exception: {exc}', flush=True)
    else:
        print('=== Phase-A Heavy Combat Smoke Disabled (ENABLE_PHASE_A_HEAVY_SMOKE=False) ===', flush=True)

    # Always guarantee submission.parquet is written and verified
    if not submission_path.exists():
        dummy = pd.DataFrame(data=[['1_0', '1', True, 1]], columns=['row_id', 'game_id', 'end_of_game', 'score'])
        dummy.to_parquet(submission_path, index=False)
    print('=== LCLD PHASE A VALIDATION COMPLETE; SUBMISSION ARTIFACT READY ===', flush=True)
else:
    print('Phase A heavy smoke skipped: this execution is a Phase B competition rerun.', flush=True)


In [ ]:
# =============================================================================
# CELL 5: PHASE-B GATEWAY & CONCURRENT GAMEPLAY EXECUTION (RERUN ONLY)
# =============================================================================
import os, sys, time, pathlib, subprocess, json, urllib.request
import pandas as pd

is_rerun = os.getenv('KAGGLE_IS_COMPETITION_RERUN', '').strip().lower() in ('1', 'true')
working_root = pathlib.Path('/kaggle/working')
submission_path = working_root / 'submission.parquet'

if not is_rerun:
    print('=== Phase B competition execution skipped (Phase A commit/dry-run mode). ===', flush=True)
else:
    print('=================================================================', flush=True)
    print('=== STARTING PHASE B ISOLATED COMPETITION RUNTIME ===', flush=True)
    print('=================================================================', flush=True)

    # 1. Setup arcade client environment and write .env
    env_path = working_root / '.env'
    arcade_settings = {
        'SCHEME': 'http',
        'HOST': 'gateway',
        'PORT': '8001',
        'ARC_API_KEY': 'test-key-123',
        'ARC_API_BASE': 'http://gateway:8001',
        'ARC_BASE_URL': 'http://gateway:8001/',
        'OPERATION_MODE': 'competition',
        'ENVIRONMENTS_DIR': '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files',
        'RECORDINGS_DIR': '/kaggle/working/server_recording',
        'LCLD_MAX_ACTIONS_PER_GAME': '500',
        'LCLD_MAX_ACTIONS_PER_LEVEL': '500',
        'LCLD_GAME_WALL_CLOCK_LIMIT_SECONDS': '5000',
        'LCLD_GAME_CONCURRENCY': '5',
        'LCLD_COMPETITION_WALL_CLOCK_LIMIT_SECONDS': '30600',
    }
    os.environ.update(arcade_settings)
    env_path.write_text(''.join(f'{k}={v}\n' for k, v in arcade_settings.items()), encoding='utf-8')
    print(f'[Phase B] Written gateway configuration to {env_path}', flush=True)

    # 2. Install vLLM wheelhouse into /kaggle/working/vllm-site-packages
    import phase_a_heavy_smoke
    wheelhouse = phase_a_heavy_smoke.find_wheelhouse_path()
    if not wheelhouse:
        raise FileNotFoundError('vLLM wheelhouse dataset not found in /kaggle/input')
    site_packages = phase_a_heavy_smoke.install_vllm_wheelhouse(wheelhouse)

    # 3. Locate model weights
    model_path = phase_a_heavy_smoke.find_model_path()
    if not model_path:
        raise FileNotFoundError('Qwen model weights not found in /kaggle/input')

    # 4. Start vLLM server with logging
    ready = phase_a_heavy_smoke.start_vllm_server(model_path, site_packages)
    if not ready:
        print('=== vLLM SERVER LOG TAIL (Startup Failure) ===', flush=True)
        print(phase_a_heavy_smoke._vllm_log_tail(30000), flush=True)
        raise RuntimeError('vLLM server failed to start within timeout')

    # 5. Gateway handshake check
    print('[Phase B] Checking gateway connectivity at http://gateway:8001/api/games...', flush=True)
    deadline = time.monotonic() + 700.0
    gateway_ready = False
    while time.monotonic() < deadline:
        try:
            req = urllib.request.Request(
                'http://gateway:8001/api/games',
                headers={'Accept': 'application/json', 'X-API-Key': os.environ.get('ARC_API_KEY', '')},
            )
            with urllib.request.urlopen(req, timeout=10) as r:
                if 200 <= r.status < 500:
                    print(f'[Phase B] Gateway handshake OK (status={r.status})', flush=True)
                    gateway_ready = True
                    break
        except Exception:
            time.sleep(4.0)
    if not gateway_ready:
        print('[Phase B] Warning: gateway handshake timed out; proceeding with caution.', flush=True)

    # 6. Execute games concurrently
    try:
        from arc_agi import Arcade
        from lcld_competition_child import run_concurrent_arcade_games
        arcade = Arcade()
        results = run_concurrent_arcade_games(arcade, concurrency=5)
        print(f'[Phase B] Completed gameplay across {len(results)} environments.', flush=True)
    except Exception as run_exc:
        print(f'[Phase B ERROR] Exception during concurrent gameplay: {run_exc}', flush=True)
        print('=== vLLM SERVER LOG TAIL (Post-Error) ===', flush=True)
        print(phase_a_heavy_smoke._vllm_log_tail(30000), flush=True)
    finally:
        phase_a_heavy_smoke.stop_vllm_server()
        if not submission_path.exists():
            dummy = pd.DataFrame(data=[['1_0', '1', True, 1]], columns=['row_id', 'game_id', 'end_of_game', 'score'])
            dummy.to_parquet(submission_path, index=False)
        print('=== PHASE B WORKFLOW COMPLETE; SUBMISSION COMMITTED ===', flush=True)
